# 🌊 03 - Seaborn: Distributions

Module 07 described a column's **shape** with numbers: center, spread, skew, outliers. This notebook draws the shape itself, and compares it across groups, straight from the raw rows.

**Seaborn** is a library built on Matplotlib for statistical charts. You name the **columns** you want on each part of the chart (`x=`, `y=`, `hue=`, `col=`), and Seaborn does the grouping, counting, and coloring for you. That's why it wants **long** data (notebook 02): one row per observation.

## ✅ Learning Goals
- Explain Seaborn's column-mapping model (`data=`, `x=`, `hue=`, ...) and **axes-level** vs. **figure-level** functions
- Draw histograms with `sns.histplot()`, and control bins, `hue`, `multiple=`, `stat=`, and `common_norm=`
- Compare groups of different sizes fairly with `stat='density'` / `'percent'` and `common_norm=False`
- Use KDE and ECDF plots, and know when each helps
- Read a box plot as Module 07's quartiles and IQR fences, and combine it with violin and strip plots
- Build small multiples with `sns.displot(col=...)`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUT = Path('03-OutputFiles')
OUT.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')       # one call styles every chart that follows

MEASUREMENTS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# Same cleaning as the Module 07 lab
penguins = pd.read_csv('penguins.csv').dropna(subset=MEASUREMENTS)
penguins['sex'] = penguins['sex'].str.title()

titanic = pd.read_csv('titanic.csv')
titanic['outcome'] = titanic['Survived'].map({0: 'died', 1: 'survived'})

## 1. The Seaborn Model
Every Seaborn function takes a DataFrame and **maps columns to roles**:

```python
sns.histplot(data=penguins, x='body_mass_g', hue='species')
#            the table      the x-axis       color by this column
```

Functions come in two levels:

| Level | Examples | Draws on | Returns | Size with |
|---|---|---|---|---|
| **Axes-level** | `histplot`, `kdeplot`, `boxplot`, `violinplot`, `scatterplot` | one `ax` (pass `ax=` to choose) | an `ax` | `plt.subplots(figsize=...)` |
| **Figure-level** | `displot`, `catplot`, `relplot` | a whole new figure, one panel per group (`col=`, `row=`) | a `FacetGrid` | `height=`, `aspect=` |

Axes-level functions slot into the `fig, ax` workflow from notebook 01. Figure-level functions build the panels for you.

## 2. Histograms with `sns.histplot()`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=penguins, x='body_mass_g', binwidth=100, ax=ax)
ax.set(title='Body Mass of All Penguins', xlabel='Body mass (g)')
plt.show()

Module 07 said body mass was roughly symmetric overall (skew ≈ 0.47). The histogram shows something the skew number couldn't: a big pile on the left, then a long, lumpy **shoulder** on the right that doesn't taper off smoothly. That's a hint that two different groups may be mixed together. (When the groups are far enough apart to make two clear peaks, the distribution is called **bimodal**.)

`binwidth=100` sets each bar to 100 g wide. You can set the number of bars with `bins=` instead. As Foundations 07 showed, the choice changes what you see, so try a few.

## 3. Groups with `hue`
`hue='species'` splits the data by species and colors each group:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=penguins, x='body_mass_g', hue='species', binwidth=100, ax=ax)
ax.set(title='Body Mass by Species', xlabel='Body mass (g)')
plt.show()

There's the second group: **Gentoos**, making up most of that right shoulder. Adelies and Chinstraps overlap almost completely. That matches the Module 07 lab's species summary (Gentoo mean ≈ 5,076 g vs. about 3,700 g for the others), but now you can see the whole distribution, not just one number per group.

Overlapping bars get hard to read. `multiple=` controls how the groups share the space:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, layout='constrained')

for ax, how in zip(axes.flat, ['layer', 'stack', 'dodge', 'fill']):
    sns.histplot(data=penguins, x='body_mass_g', hue='species', binwidth=200,
                 multiple=how, ax=ax, legend=(how == 'layer'))
    ax.set(title=f"multiple='{how}'", xlabel='Body mass (g)')

plt.show()

| `multiple=` | Shows | Good for |
|---|---|---|
| `'layer'` (default) | groups drawn on top of each other, semi-transparent | 2–3 groups that don't overlap much |
| `'stack'` | groups piled up | the **total**, plus each group's part of it |
| `'dodge'` | bars side by side within each bin | comparing groups bin by bin |
| `'fill'` | each bin stretched to 100% | the **mix** of groups at each value |

`element='step'` (outlines instead of filled bars) is another fix for overlap.

## 4. Fair Comparisons: `stat=` and `common_norm=`
Module 07 notebook 04: when groups have different sizes, compare **rates**, not counts. Histograms have the same problem. Compare the ages of Titanic passengers who survived and who died:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout='constrained')

sns.histplot(data=titanic, x='Age', hue='outcome', binwidth=5, element='step', ax=axes[0])
axes[0].set(title='Counts: more people died, so the "died" shape dominates')

sns.histplot(data=titanic, x='Age', hue='outcome', binwidth=5, element='step',
             stat='percent', common_norm=False, ax=axes[1])
axes[1].set(title='Percent within each group: the shapes can be compared')

plt.show()

- `stat='percent'` (or `'density'`) puts the y-axis on a relative scale instead of raw counts.
- `common_norm=False` makes **each group** add up to 100% on its own. With the default (`True`), the percentages are of everyone combined, and the bigger group still looks bigger.

In the right-hand chart, survivors are noticeably more common in the youngest bins (children). The left-hand chart nearly hides that.

✅ **Your Turn:** Make the right-hand chart for `Fare` instead of `Age`, using a log x-axis (`log_scale=True`, a `histplot` argument). Which group paid more?

In [ ]:
# Your turn here

## 5. Smooth Shapes: KDE Plots
A **kernel density estimate (KDE)** draws a smooth curve instead of bars. It's easier to compare several groups when the curves overlap:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=penguins, x='flipper_length_mm', hue='species', fill=True, alpha=0.3,
            common_norm=False, ax=ax)
ax.set(title='Flipper Length by Species', xlabel='Flipper length (mm)')
plt.show()

Two cautions:
- **Smoothing is a choice.** `bw_adjust=` controls it: lower is more wiggly, higher is smoother. Like bin width, too much hides real detail.
- **A KDE can spill past what's possible**, such as negative ages or fares. Add `cut=0` to stop the curve at the data's actual min and max.

`sns.histplot(..., kde=True)` draws the curve on top of the bars, if you want both.

## 6. Reading Percentiles Directly: ECDF Plots
An **ECDF** (empirical cumulative distribution function) plots, for every value, **the share of data at or below it**. No bins, no smoothing, every point shown. You can read Module 07's percentiles straight off the y-axis:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.ecdfplot(data=penguins, x='body_mass_g', hue='species', ax=ax)
ax.axhline(0.5, color='gray', linestyle=':')
ax.set(title='ECDF of Body Mass: Read the Median Where Each Curve Crosses 0.5',
       xlabel='Body mass (g)', ylabel='Share of penguins at or below')
plt.show()

Where a species' curve crosses the dotted 0.5 line is its **median**. A curve further right means a heavier species. A steeper curve means less spread.

## 7. Box Plots, Revisited
A box plot is Module 07 notebook 03 drawn as a picture:

```text
   o            ← outlier: beyond the fence
   ┬            ← whisker: the most extreme value INSIDE the fence
 ┌─┴─┐
 │   │          ← Q3 (75th percentile)
 ├───┤          ← median
 │   │          ← Q1 (25th percentile)
 └─┬─┘          box height = IQR
   ┴
```

The fences are exactly the `iqr_fences()` from Module 07. Check that on Titanic fares in 1st class:

In [ ]:
def iqr_fences(series, k=1.5):
    """Return the (lower, upper) outlier fences for a numeric Series."""
    q1, q3 = series.quantile([0.25, 0.75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


first = titanic[titanic['Pclass'] == 1]['Fare']
low, high = iqr_fences(first)

fig, ax = plt.subplots(figsize=(5, 5))
sns.boxplot(y=first, ax=ax, color='lightgray')
ax.axhline(high, color='tab:red', linestyle='--', label=f'upper fence = £{high:.0f}')
ax.set(title='1st-Class Fares', ylabel='Fare (£)')
ax.legend(loc='center right')
plt.show()

print('Points beyond the fence:', (first > high).sum())

Every dot above the red line is a value `iqr_fences()` would flag, and the whisker stops at the last value inside the fence.

### Comparing groups
Box plots shine for comparing **many** groups side by side. `order=` sets the order of the boxes (Module 07 notebook 01: order means something, so choose it on purpose):

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=penguins, x='species', y='body_mass_g', hue='sex',
            order=['Adelie', 'Chinstrap', 'Gentoo'], ax=ax)
ax.set(title='Body Mass by Species and Sex', xlabel='', ylabel='Body mass (g)')
plt.show()

## 8. What a Box Hides: Violin and Strip Plots
A box plot reduces each group to five numbers, so it can't show a second peak, or how many points there are. Two complements:
- **Violin plot**: a KDE turned sideways and mirrored, so it shows the full shape.
- **Strip plot**: every individual point, with a little random sideways **jitter** so they don't sit on top of each other.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True, layout='constrained')

sns.violinplot(data=penguins, x='species', y='flipper_length_mm', inner='quart',
               hue='species', legend=False, ax=axes[0])
axes[0].set(title="Violin (inner='quart' draws the quartiles)", xlabel='', ylabel='Flipper length (mm)')

sns.boxplot(data=penguins, x='species', y='flipper_length_mm', color='white',
            showfliers=False, ax=axes[1])
sns.stripplot(data=penguins, x='species', y='flipper_length_mm', hue='species',
              alpha=0.5, jitter=0.25, legend=False, ax=axes[1])
axes[1].set(title='Box + every point (strip)', xlabel='')

fig.savefig(OUT / 'flipper_violin_strip.png', dpi=150, bbox_inches='tight')
plt.show()

`showfliers=False` hides the box plot's own outlier dots, because the strip plot already draws every point (otherwise outliers would appear twice).

| Chart | Shows | Best when |
|---|---|---|
| Box | 5-number summary + outliers | many groups, quick comparison |
| Violin | full shape (peaks, skew) | shape matters, groups are large |
| Strip / swarm | every point | small to medium groups, and you want to see n |
| Box + strip | summary **and** every point | a clear default for a few groups |

## 9. Small Multiples with `sns.displot()`
`hue` puts all groups on one chart. When they overlap too much, give each group its **own panel** instead. Figure-level `displot` does it with `col=` (and `row=`):

In [ ]:
g = sns.displot(data=penguins, x='bill_length_mm', col='species', row='sex',
                binwidth=1, height=2.8, aspect=1.3, color='tab:blue')
g.set_axis_labels('Bill length (mm)', 'Penguins')
g.set_titles('{col_name} · {row_name}')
g.figure.suptitle('Bill Length by Species and Sex', y=1.03)
g.savefig(OUT / 'bill_length_grid.png', dpi=150, bbox_inches='tight')
plt.show()

Every panel shares the same axes, so you can compare any two panels directly (notebook 01's `sharey=True`, done for you). Figure-level functions are sized with `height=` (inches per panel) and `aspect=` (width ÷ height), not `figsize`. The returned `FacetGrid` has its own methods: `set_axis_labels`, `set_titles`, `savefig`, and `g.figure` for the whole figure.

---
### 📝 Summary
- Seaborn maps **columns to roles** (`x`, `y`, `hue`, `col`, `row`) on **long** data.
- Axes-level functions (`histplot`, `boxplot`, ...) take `ax=`. Figure-level ones (`displot`, `catplot`, `relplot`) build a grid of panels.
- `histplot`: choose `binwidth`/`bins`, separate groups with `multiple=`, and compare groups of different sizes with `stat='percent'` + `common_norm=False`.
- KDE for smooth comparison (watch `bw_adjust` and `cut=0`). ECDF for reading percentiles.
- A box plot **is** the IQR fence rule. Violins show shape, and strips show every point.
- Overlapping `hue` groups? Use small multiples with `col=`/`row=`.

## 🏋️ Practice

### Practice 1 — Bill Depth by Species (easy)
Make a `histplot` of `bill_depth_mm` with `hue='species'` and `element='step'`. Which species stands apart, and in which direction? Does that match the Module 07 lab's species summary?

In [ ]:
# Your code here

### Practice 2 — Pokémon Totals (medium)
Load `pokemon.csv`. Draw `Total` by `Legendary` two ways, side by side in one figure: (a) a histogram with **counts**, and (b) a histogram with `stat='density', common_norm=False`. In a Markdown cell, explain why the Legendary distribution is nearly invisible in (a).

In [ ]:
# Your code here

*Your explanation here.*

### Practice 3 — Box vs. Violin vs. Strip (harder)
For Titanic `Age` by `class` (create it with `.map()`), draw a 1 × 3 figure: box plot, violin plot, and box + strip. Use the same `order=['1st', '2nd', '3rd']` in every panel. Which chart makes it clearest that 3rd class had many more passengers? Which shows the cluster of young children best?

In [ ]:
# Your code here

*Your answer here.*

## ✏️ Your Turn
Use `sns.displot()` to draw `body_mass_g` with `col='island'` and `hue='species'`. Save it as `OUT / 'mass_by_island.png'`. Then, in a Markdown cell, connect it to the Module 07 lab: why would comparing "island" averages be misleading?

In [ ]:
# Your turn here

*Your explanation here.*

## 🔥 Challenge (Optional) — Bin Width Experiment
Write a loop that draws `body_mass_g` (all penguins, no `hue`) in a 1 × 4 figure with `binwidth` of 25, 100, 250, and 800. At which width is the right shoulder (the Gentoos) easiest to see, and at which does it disappear? Add a fifth panel with `sns.kdeplot(..., bw_adjust=...)` set so smooth that it hides the shoulder. What does that tell you about trusting a single chart?

In [ ]:
# Your code here

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `03-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
✅ Next up → [Seaborn: Relationships and Categories](04-Seaborn_Relationships_and_Categories.ipynb): scatter plots, trend lines, heatmaps, and the penguin paradox from Module 07, made visible.